In [9]:
import sys
sys.path.append("..")

from src.solvers.bit_manipulation import BitManipulationSolver
from src.solvers.equations import EnsembleEquationsSolver
from src.solvers.encryption import EncryptionSolver
from src.solvers.unit_conversion import UnitConversionSolver
from src.solvers.gravitational import GravitationalSolver
from src.solvers.numeral_system import NumeralSystemSolver

In [10]:
import re
import pandas as pd
import statistics
from tqdm import tqdm

tqdm.pandas()


In [11]:
data = pd.read_csv("../data/raw/train.csv")

In [12]:
data["prompt_eda"] = data.prompt.str.split('.').apply(lambda x: x[0])

In [13]:
task_classes = {
    "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers":  "bit manipulation",
    "In Alice's Wonderland, secret encryption rules are used on text": "encryption",
    "In Alice's Wonderland, numbers are secretly converted into a different numeral system": "conversion to diff numeral system",
    "In Alice's Wonderland, a secret unit conversion is applied to measurements": "unit conversion",
    "In Alice's Wonderland, the gravitational constant has been secretly changed": "gravitational",
    "In Alice's Wonderland, a secret set of transformation rules is applied to equations": "equations transformation"
}

In [14]:
data["label"] = data.prompt_eda.map(task_classes)
data["label"].value_counts()

label
bit manipulation                     1602
gravitational                        1597
unit conversion                      1594
encryption                           1576
conversion to diff numeral system    1576
equations transformation             1555
Name: count, dtype: int64

In [15]:
from pandarallel import pandarallel

pandarallel.initialize(nb_workers=24, progress_bar=True)


INFO: Pandarallel will run on 24 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [16]:
eq_df = data[data['label'] == 'equations transformation'].copy().sample(100)

# Инициализируем наш решатель
solver = EnsembleEquationsSolver()

print("--- Starting First Run (Zero-Shot) ---")
# Пытаемся решить честно, без подсказок
eq_df['generated_cot'] = eq_df['prompt'].parallel_apply(lambda x: solver.generate_cot(x))
eq_df['computed_answer'] = eq_df['generated_cot'].parallel_apply(solver.extract_answer)

# Считаем количество провалов
eq_df['is_correct_pass1'] = eq_df['computed_answer'] == eq_df['answer'].astype(str).str.strip()
failed_mask = ~eq_df['is_correct_pass1']
print(f"Fail on first run: {failed_mask.sum()} rows out of {len(eq_df)}\n")
print(f"Fail on first run: {eq_df['is_correct_pass1'].mean() * 100}")

# Второй проход (с подсказкой для генерации обучающего CoT)
if failed_mask.sum() > 0:
    print("--- Starting Second Run (Reverse-Engineering CoT for Failed Rows) ---")
    
    def solve_with_fallback(row):
        # Передаем правильный ответ в качестве подсказки (Oracle)
        return solver.generate_cot(row['prompt'], answer_hint=str(row['answer']).strip())

    # Применяем fallback только к тем строкам, которые не решились сами
    eq_df.loc[failed_mask, 'generated_cot'] = eq_df[failed_mask].parallel_apply(solve_with_fallback, axis=1)
    
    # Снова извлекаем ответ из нового CoT
    eq_df.loc[failed_mask, 'computed_answer'] = eq_df.loc[failed_mask, 'generated_cot'].parallel_apply(solver.extract_answer)

# Итоговая проверка качества
eq_df['is_correct_final'] = eq_df['computed_answer'] == eq_df['answer'].astype(str).str.strip()
final_accuracy = eq_df['is_correct_final'].mean() * 100

print(f"Final SFT Dataset Consistency (Accuracy): {final_accuracy:.2f}%")

--- Starting First Run (Zero-Shot) ---


Fail on first run: 68 rows out of 100

Fail on first run: 32.0
--- Starting Second Run (Reverse-Engineering CoT for Failed Rows) ---


Final SFT Dataset Consistency (Accuracy): 47.00%


In [20]:
target_labels = ['equations transformation']


filtered_data = eq_df[(eq_df['label'].isin(target_labels)) & (eq_df["is_correct_final"] == False)]

sampled_data = filtered_data.groupby('label').sample(n=50, random_state=42).reset_index(drop=True)

for index, row in sampled_data.iterrows():
    print(f"=== Категория: {row['label']} | ID: {row['id']} ===")
    print("--- Промпт (начало) ---")
    print(str(row['prompt']) + "...\n")
    
    print("--- Сгенерированный CoT ---")
    cot_val = row.get('generated_cot', row.get('generated cot', 'Отсутствует'))
    print(cot_val)
    
    print("\n--- Вычисленный ответ ---")
    ans_val = row.get('computed_answer', row.get('computed answer', 'Отсутствует'))
    print(ans_val)
    print("\n--- Истинный ответ ---")
    ans_val = row.get('answer', row.get('computed answer', 'Отсутствует'))
    print(ans_val)
    
    print("\n" + "="*80 + "\n")

=== Категория: equations transformation | ID: d98d85c0 ===
--- Промпт (начало) ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
(#-`> = ''
|'-'# = ||
@&+'> = '&&&
Now, determine the result for: (@+@?...

--- Сгенерированный CoT ---
Let's systematically analyze the examples to discover the hidden transformation rule.
The symbols represent standard base-10 mathematical operations.
Testing standard arithmetic reveals contradictions. For instance, the outputs in the examples do not match standard math.
The operation might be applied digit-by-digit rather than on whole numbers.
Digit-wise operations do not yield a consistent rule across all examples.
This could be a cryptarithm where symbols or letters map to specific base-10 digits.
Analysis shows no consistent character-to-digit mapping satisfies all equations simultaneously. Let's pivot to a non-mathematical structure.
The operators might be string manipulation functions (